In [2]:
%load_ext autoreload
%autoreload 2

# Import library

In [3]:

from sentence_transformers import SentenceTransformer, util , SimilarityFunction


import pandas as pd
import numpy as np
import regex as re
import itertools as it
import matplotlib.pyplot as plt
import pickle
import os
import torch
import seaborn as sns 
import sys



/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
2024-05-29 17:25:39.719609: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
# Add the src directory to the sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
print(sys.path)
import data_io

['/Library/Frameworks/Python.framework/Versions/3.12/lib/python312.zip', '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12', '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/lib-dynload', '', '/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages', '/Users/gl665/PycharmProjects/Phrame_MentalMaps/src']


# Read data


## SYNCH Project 

In [5]:
new_dir = "/Users/gl665/PycharmProjects/Phrame_MentalMaps"
# Change the current working directory
os.chdir(new_dir)


my_data_RC = {}
folder_path = 'results/SYNCH/JSON_file/RC/'

cld_data = data_io.read_json(folder_path, my_data_RC)

load: FM 51220Clean.json
load: Passages Clean.json
load: full circle GBM 5 24 21clean.json
load: C 7 6 20 clean model.json
load: Wavi combined clean.json
load: drug justice model clean.json
load: DSS23.json
load: E clean model 572020.json
load: nat model clean2.json
load: FC Prov Clean.json
load: full circle clean.json
load: reentry clean model.json
load: DV justice model clean.json


In [6]:
# clean variables 

def clean_variable_name(variable_name):
    cleaned_name = variable_name.replace("\\", "")
    cleaned_name = cleaned_name.replace("/", " or ")
    cleaned_name = cleaned_name.lower()

    return cleaned_name

for k, v in cld_data.items():
    for key, value in v[0].items():
        value = clean_variable_name(value)

# Find semantic similary variable names 

In [7]:
variable_names = []

for k, v in cld_data.items():
    cld_new_vars_name = []
    cld_var_names = list(v[0].values())

    for name in cld_var_names:
        name = clean_variable_name(name)
        cld_new_vars_name.append(name)

    variable_names.append(cld_new_vars_name)

variable_names = sum(variable_names, [])

print("number of originial nodes: ", len(variable_names))

variable_names = np.sort(list(set(variable_names)))


print("number of originial unique nodes: ", len(variable_names))
# print all list of variable names


number of originial nodes:  1166
number of originial unique nodes:  1021


In [ ]:
for i in variable_names: 
    print(i)

In [143]:
# Embed some sentences
# variable_names = [
#     'customer_age', 'client_age', 'age_of_customer', 
#     'customer_id', 'client_id', 'id_of_customer', 
#     'purchase_date', 'date_of_purchase', 'transaction_date', 'customer__age', 
# 'parental alcohol use', 'maternal alcohol use', 'parents drinking in home', 'paternal alcohol use', 'youth alcohol use',
# 'partner control', 'partner control or power', 'partner dependence',
# 'partner sexual decision making', 'sexual activity decision making ability', 'sexual behavior decision-making',
# 'perception of resillilence to relapse', 'relapse', 'resistance or resilience to relapse',
# 'perceptions of police as promoting safety', 'trust in police',
# 'percieved efficacy of treatment', 'treatment program efficacy', 'treatment success'

# ]

In [8]:
# Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(variable_names)

similarities = model.similarity(embeddings, embeddings)

# TODO: Finetune the sentence transformer model to improve performance of the model. 
# # Change the similarity function to Manhattan distance
# model.similarity_fn_name = SimilarityFunction.MANHATTAN
# print(model.similarity_fn_name)
# # => "manhattan"

# similarities = model.similarity(embeddings, embeddings)
# print(similarities)


/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# Output the pairs with their score
for idx_i, sentence1 in enumerate(variable_names):
    print(sentence1)
    for idx_j, sentence2 in enumerate(variable_names):
        print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")
        break 

In [9]:
# Threshold for considering names as similar
threshold = 0.75

# Group semantically similar names
groups = []
visited = set()

for idx, name in enumerate(variable_names):
    if name not in visited:
        similar_names = [name]
        for jdx, similarity in enumerate(similarities[idx]):


            if jdx != idx and similarity >= threshold and variable_names[jdx] not in visited:
                similar_names.append(variable_names[jdx])
                visited.add(variable_names[jdx])
                
        groups.append(similar_names)
        visited.update(similar_names)


In [10]:
print("numbers of final variables after grouping: ", len(groups))


numbers of final variables after grouping:  738


In [11]:
# Print grouped variable names
for i, group in enumerate(groups):
    # if len(group) > 1:
    print(f"Group {i + 1}: {group}")

len(groups)

Group 1: ['abandonment']
Group 2: ['ability for health insurance to pay for care', 'ability to pay for treatment', 'access to health insurance']
Group 3: ['ability for victims to recieve housing resources', 'availability of victim shelter housing']
Group 4: ['ability to address trauma']
Group 5: ['ability to afford birth control', 'access to birth control', 'affordability of birth control']
Group 6: ['ability to be self sustaining', 'self-sufficiency', 'self-sustainment']
Group 7: ['ability to fufil dss requests']
Group 8: ['ability to fufill court requirements', 'court based requirements', 'court requirements fufilled']
Group 9: ['ability to move from detox to treatment']
Group 10: ['ability to navigate colonial spaces']
Group 11: ['ability to pay for tx']
Group 12: ['ability to qualify for extra ss help']
Group 13: ['ability to schedule  or  time appointment']
Group 14: ['ability to self pay']
Group 15: ['ability to sustain self or kids']
Group 16: ['acceptability of birth control', 

738

In [ ]:
# create a dictionary to swap the words that are bundled in the group. 
swap_dict = {}

for group in groups:
    if len(group) > 1:
        swap_dict[group[0]] = group
swap_dict

swap_dict_reversed = {v: k for k, vv in swap_dict.items() for v in vv}
swap_dict_reversed


In [13]:
len(swap_dict_reversed)

455

In [23]:
# Swap the variable names in the data

cld_data_new = cld_data.copy()

for k, v in cld_data_new.items():
    for key, value in v[0].items():
        if value in swap_dict_reversed:
            v[0][key] = swap_dict_reversed[value]



In [24]:
variable_names = [list(v[0].values()) for k, v in cld_data_new.items()]

variable_names = sum(variable_names, [])

variable_names = np.sort(list(set(variable_names)))


print("number of originial unique nodes: ", len(variable_names))


number of originial unique nodes:  755


In [29]:
# import pickle

# # Specify the file path where you want to save the pickled data
# file_path = "results/SYNCH/JSON_file/RC_cld_data_merged.json"

# # Pickle dump the cld_data_new dictionary
# with open(file_path, "wb") as file:
#     pickle.dump(cld_data_new, file)
